In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1738, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 1738 (delta 94), reused 0 (delta 0), pack-reused 1613 (from 2)
Receiving objects: 100% (1738/1738), 628.22 MiB | 52.75 MiB/s, done.
Resolving deltas: 100% (1097/1097), done.
Updating files: 100% (232/232), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.0 MB/s eta 0:00:00


In [3]:
! pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831127 sha256=9c7e91ab31e63094b0d2835efc5cdaf5cda2b4bc6250b1897d0c27d076d27855
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [4]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [6]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [7]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [8]:
db_name = 'lightfm_study.db'
json_name = 'lightfm_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '20',
    'n_trials': 1
}

In [9]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [10]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [11]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [12]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [13]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.FactorizationMachines.LightFMRecommender import LightFMCFRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [14]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [15]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'lightfm_Optimization')


In [16]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_lightfm(optuna_trial):
    recommender_instance = LightFMCFRecommender(URM_train)

    params = {
        'n_components': optuna_trial.suggest_int("n_components", 10, 128),
        'item_alpha': optuna_trial.suggest_float("item_alpha", 1e-6, 1e-2, log=True),
        'user_alpha': optuna_trial.suggest_float("user_alpha", 1e-6, 1e-2, log=True),
        'learning_rate': optuna_trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
        'loss': optuna_trial.suggest_categorical("loss", ['bpr', 'warp']),
        'sgd_mode': optuna_trial.suggest_categorical("sgd_mode", ['adagrad', 'adadelta'])
    }

    recommender_instance.fit(**params,
                             epochs=200,
                             evaluator_object=evaluator,
                             stop_on_validation=True,
                             validation_every_n=5,
                             validation_metric="RECALL",
                             lower_validations_allowed=5)

    result_df, _ = evaluator.evaluateRecommender(recommender_instance)

    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [17]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='lightfm_Optimization',
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_lightfm, n_trials=config['n_trials'])

[I 2026-01-03 10:16:15,507] Using an existing study with name 'lightfm_Optimization' instead of creating a new one.


Previous trials found: 24
LightFMCFRecommender: Epoch 1 of 200. Elapsed time 6.34 sec
LightFMCFRecommender: Epoch 2 of 200. Elapsed time 12.58 sec
LightFMCFRecommender: Epoch 3 of 200. Elapsed time 18.84 sec
LightFMCFRecommender: Epoch 4 of 200. Elapsed time 25.03 sec
LightFMCFRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 1.83 min. Users per second: 246
LightFMCFRecommender: CUTOFF: 20 - PRECISION: 0.1404681, PRECISION_RECALL_MIN_DEN: 0.2090466, RECALL: 0.1702098, MAP: 0.0572948, MAP_MIN_DEN: 0.0809891, MRR: 0.4028751, NDCG: 0.1983158, F1: 0.1539154, HIT_RATE: 0.8699893, ARHR_ALL_HITS: 0.6478727, NOVELTY: 0.0260556, AVERAGE_POPULARITY: 0.4713789, DIVERSITY_MEAN_INTER_LIST: 0.7949513, DIVERSITY_HERFINDAHL: 0.9897461, COVERAGE_ITEM: 0.1223992, COVERAGE_ITEM_HIT: 0.0889654, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.8690902, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0171956, SHANNON_ENTROPY: 7.2891121, RATIO_DIVERSITY_HERFINDAH

[I 2026-01-03 11:51:46,740] Trial 24 finished with value: 0.2397943898094386 and parameters: {'n_components': 127, 'item_alpha': 0.0001007237052301578, 'user_alpha': 1.0780046031923186e-06, 'learning_rate': 0.015153814758693429, 'loss': 'warp', 'sgd_mode': 'adagrad'}. Best is trial 21 with value: 0.24916323482339714.


CPU times: user 2h 33min 7s, sys: 45 s, total: 2h 33min 52s
Wall time: 1h 35min 32s


In [18]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

Updating JSON. Best: 0.249163 (Prev: 0.107505)
